# 02 — TCR Embedding (CNN Autoencoder)

Trenira CNN avtoencoder na CDR3β sekvencah in producira:
- `trained_tcr_embeddings_ae.pkl` — embeddingi za vsako unikatno TCR sekvenco `(n_tcrs, 100)`
- `data_tcr.pkl` — embeddingi za vsako celico `(n_cells, 100)`, ki jih `trim.py` potrebuje

## 0. Namestitev in mount

In [ ]:
!pip install -q umap-learn logomaker

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Uvozi in poti

In [ ]:
import numpy as np
import pandas as pd
import scipy.io
import sklearn.svm
import sklearn.cluster
import sklearn.metrics
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import sys
import time
import math
import logomaker as lm
import os
import pickle
import torch
from torch import nn
import torch.nn.functional as F

data_path = '/content/drive/MyDrive/Diploma/data/processed'
pca_file_path = os.path.join(data_path, 'data_rna_pca.pkl')

TRAINING_STEPS = 50100
output_path = os.path.join(data_path, f'tcr_ae/step_{TRAINING_STEPS}')
os.makedirs(output_path, exist_ok=True)
print('output_path:', output_path)

fig = plt.figure()

## 2. Pomožne funkcije in razredi

In [ ]:
class Loader(object):
    """Batch loader za numpy matrike."""

    def __init__(self, data, labels=None, shuffle=False):
        self.start = 0
        self.epoch = 0
        self.data = [x for x in [data, labels] if x is not None]
        self.labels_given = labels is not None
        if shuffle:
            self.r = list(range(data.shape[0]))
            np.random.shuffle(self.r)
            self.data = [x[self.r] for x in self.data]

    def next_batch(self, batch_size=100):
        num_rows = self.data[0].shape[0]
        if self.start + batch_size < num_rows:
            batch = [x[self.start:self.start + batch_size] for x in self.data]
            self.start += batch_size
        else:
            self.epoch += 1
            batch_part1 = [x[self.start:] for x in self.data]
            batch_part2 = [x[:batch_size - (x.shape[0] - self.start)] for x in self.data]
            batch = [np.concatenate([x1, x2], axis=0) for x1, x2 in zip(batch_part1, batch_part2)]
            self.start = batch_size - (num_rows - self.start)
        if not self.labels_given:
            return batch[0]
        else:
            return batch

    def iter_batches(self, batch_size=100):
        num_rows = self.data[0].shape[0]
        end = 0
        if batch_size > num_rows:
            if not self.labels_given:
                yield [x for x in self.data][0]
            else:
                yield [x for x in self.data]
        else:
            for i in range(num_rows // batch_size):
                start = i * batch_size
                end = (i + 1) * batch_size
                if not self.labels_given:
                    yield [x[start:end] for x in self.data][0]
                else:
                    yield [x[start:end] for x in self.data]
            if end < num_rows:
                if not self.labels_given:
                    yield [x[end:] for x in self.data][0]
                else:
                    yield [x[end:] for x in self.data]


def get_atchley():
    """Atchleyevi faktorji: 5 numeričnih lastnosti za vsako aminokislino."""
    atchley = pd.DataFrame(
            [[' ', 0, 0, 0, 0, 0],
            ['A', -0.59145974, -1.30209266, -0.7330651,  1.5703918, -0.14550842],
            ['C', -1.34267179,  0.46542300, -0.8620345, -1.0200786, -0.25516894],
            ['D',  1.05015062,  0.30242411, -3.6559147, -0.2590236, -3.24176791],
            ['E',  1.35733226, -1.45275578,  1.4766610,  0.1129444, -0.83715681],
            ['F', -1.00610084, -0.59046634,  1.8909687, -0.3966186,  0.41194139],
            ['G', -0.38387987,  1.65201497,  1.3301017,  1.0449765,  2.06385566],
            ['H',  0.33616543, -0.41662780, -1.6733690, -1.4738898, -0.07772917],
            ['I', -1.23936304, -0.54652238,  2.1314349,  0.3931618,  0.81630366],
            ['K',  1.83146558, -0.56109831,  0.5332237, -0.2771101,  1.64762794],
            ['L', -1.01895162, -0.98693471, -1.5046185,  1.2658296, -0.91181195],
            ['M', -0.66312569, -1.52353917,  2.2194787, -1.0047207,  1.21181214],
            ['N',  0.94535614,  0.82846219,  1.2991286, -0.1688162,  0.93339498],
            ['P',  0.18862522,  2.08084151, -1.6283286,  0.4207004, -1.39177378],
            ['Q',  0.93056541, -0.17926549, -3.0048731, -0.5025910, -1.85303476],
            ['R',  1.53754853, -0.05472897,  1.5021086,  0.4403185,  2.89744417],
            ['S', -0.22788299,  1.39869991, -4.7596375,  0.6701745, -2.64747356],
            ['T', -0.03181782,  0.32571153,  2.2134612,  0.9078985,  1.31337035],
            ['V', -1.33661279, -0.27854634, -0.5440132,  1.2419935, -1.26225362],
            ['W', -0.59533918,  0.00907760,  0.6719274, -2.1275244, -0.18358096],
            ['Y',  0.25999617,  0.82992312,  3.0973596, -0.8380164,  1.51150958]])
    atchley = atchley.set_index(0)
    return atchley


def numpy2torch(x, type=torch.FloatTensor):
    return torch.from_numpy(x).type(type).to(device)


def check_for_nan(model):
    for name, param in model.named_parameters():
        flag = False
        if param.grad is not None and torch.isnan(param.grad).any():
            for name, param in model.named_parameters():
                if param.grad is not None and torch.isnan(param.grad).any():
                    print(name)
            flag = True
        if flag:
            raise Exception('NaN v gradientih!')


def get_x_tcr_from_label(labels_matrix):
    """Za batch labelov vrne Atchley faktorje za vsako TCR sekvenco."""
    return np.take(get_atchley().values,
                   np.take(df_all_tcrs_array, labels_matrix[:, col_tcr].astype(np.int32), axis=0),
                   axis=0)


print('Pomožne funkcije definirane.')

## 3. Arhitektura CNN avtoenkoderja

In [ ]:
class CNN(nn.Module):
    """Encoder: 3x Conv1d (stride=2) + FC → embedding 100 dim."""
    def __init__(self, **kwargs):
        super().__init__()
        hdim = kwargs['nbase']
        dim_in = kwargs['dim_in']
        dim_out = kwargs['dim_out']
        dim_len = kwargs['dim_len']
        strides = [2, 2, 2]
        ksize = 3
        self.conv1 = torch.nn.Conv1d(dim_in, hdim // 1, kernel_size=ksize, stride=strides[0], padding=(ksize - 1) // 2)
        self.conv2 = torch.nn.Conv1d(hdim // 1, hdim // 2, kernel_size=ksize, stride=strides[1], padding=(ksize - 1) // 2)
        self.conv3 = torch.nn.Conv1d(hdim // 2, hdim // 4, kernel_size=ksize, stride=strides[2], padding=(ksize - 1) // 2)
        self.bn1 = nn.BatchNorm1d(hdim // 1)
        self.bn2 = nn.BatchNorm1d(hdim // 2)
        self.bn3 = nn.BatchNorm1d(hdim // 4)
        sum_of_strides = sum([s > 1 for s in strides])
        self.fc_out = nn.Linear(in_features=(hdim // 4) * math.ceil(dim_len / (2**sum_of_strides)), out_features=dim_out)
        self.lrelu = torch.nn.LeakyReLU()
        self.first = True

    def forward(self, x):
        x = x.permute([0, 2, 1])
        h1 = self.lrelu(self.bn1(self.conv1(x)))
        h2 = self.lrelu(self.bn2(self.conv2(h1)))
        h3 = self.lrelu(self.bn3(self.conv3(h2)))
        h3_flat = h3.view([x.shape[0], -1])
        out = self.fc_out(h3_flat)
        if self.first:
            print('CNN oblike slojev:', h1.shape, h2.shape, h3.shape, out.shape)
            self.first = False
        return out


class CNN_T(nn.Module):
    """Decoder: FC + 2x ConvTranspose1d → rekonstrukcija Atchley faktorjev."""
    def __init__(self, **kwargs):
        super().__init__()
        hdim = self.hdim = kwargs['nbase']
        dim_in = self.dim_in = kwargs['dim_in']
        dim_out = self.dim_out = kwargs['dim_out']
        dim_len = self.dim_len = kwargs['dim_len']
        self.dim_first_len = (dim_len // 4 + 1)
        self.dim_reshape = (hdim // 4) * self.dim_first_len
        self.fc_in1 = nn.Linear(in_features=dim_in, out_features=self.dim_reshape)
        self.fc_out1 = nn.Linear(in_features=hdim // 1, out_features=dim_out)
        ksize = 3
        self.conv1 = torch.nn.ConvTranspose1d(hdim // 4, hdim // 2, kernel_size=ksize, stride=2, padding=(ksize - 1) // 2, output_padding=1)
        self.conv2 = torch.nn.ConvTranspose1d(hdim // 2, hdim // 1, kernel_size=ksize, stride=2, padding=(ksize - 1) // 2, output_padding=1)
        self.bn_in1 = nn.BatchNorm1d(self.dim_reshape)
        self.bn1 = nn.BatchNorm1d(hdim // 2)
        self.bn2 = nn.BatchNorm1d(hdim // 1)
        self.lrelu = torch.nn.LeakyReLU()

    def forward(self, x):
        h1 = self.lrelu(self.bn_in1(self.fc_in1(x)))
        h1 = h1.view([x.shape[0], self.hdim // 4, self.dim_first_len])
        h2 = self.lrelu(self.bn1(self.conv1(h1)))
        h3 = self.lrelu(self.bn2(self.conv2(h2)))
        out = h3[:, :, :self.dim_len].permute([0, 2, 1])
        out = self.fc_out1(out)
        return out


class TCR_Embedder(nn.Module):
    """CNN avtoencoder: CDR3β (Atchley, seq_len x 5) → embedding 100 dim → rekonstrukcija."""
    def __init__(self, **kwargs):
        super().__init__()
        nbase = kwargs['nbase']
        dim_in = kwargs['dim_in']
        dimz = kwargs['dimz']
        self.encoder_tcr = CNN(dim_in=dim_in, dim_out=dimz, dim_len=tcr_max_len, nbase=nbase)
        self.decoder_tcr = CNN_T(dim_in=dimz, dim_out=dim_in, dim_len=tcr_max_len, nbase=nbase)
        self.lookup_vocab = torch.nn.Embedding(len(vocab), n_embedding_vocab).to(device)
        self.lrelu = torch.nn.LeakyReLU()

    def forward(self, x):
        embedding = self.encoder_tcr(x)
        recon = self.decoder_tcr(embedding)
        return embedding, recon


print('Arhitektura definirana.')

## 4. Naloži podatke

In [ ]:
with open(os.path.join(data_path, 'data_rna.pkl'), 'rb') as f:
    data_rna = pickle.load(f)

with open(os.path.join(data_path, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)

with open(os.path.join(data_path, 'data_labels_str.pkl'), 'rb') as f:
    data_labels_str = pickle.load(f)

with open(os.path.join(data_path, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)

# Zgradi vocab iz TCR sekvenc (aminokisline + presledek za padding)
vocab = set()
[[vocab.add(c) for c in l] for l in df_all_tcrs.index]
vocab_char2num = {v: i for i, v in enumerate(sorted(vocab))}
vocab_num2char = {i: v for i, v in enumerate(sorted(vocab))}
df_all_tcrs_array = np.array([[vocab_char2num[char] for char in i] for i in df_all_tcrs.index])
tcr_max_len = df_all_tcrs_array.shape[1]

print(f'Vocab ({len(vocab)} znakov):', sorted(vocab))
print(f'TCR matrika: {df_all_tcrs_array.shape}  (n_unikatnih_tcr × max_dolzina)')

# Stolpci data_labels
col_bloodtumor = data_labels.columns.get_loc('Tissue')
col_prepost    = data_labels.columns.get_loc('Treatment Stage')
col_celltype   = data_labels.columns.get_loc('SubCellType')
col_patient    = data_labels.columns.get_loc('Patient')
col_tcr        = data_labels.columns.get_loc('CDR3(Beta1)')

print(f'data_rna:    {data_rna.shape}')
print(f'data_labels: {data_labels.shape}')
print(f'df_all_tcrs: {df_all_tcrs.shape}')

## 5. TruncatedSVD (PCA za sparse matriko)

RNA matrika je sparse — navadna PCA bi zahtevala ~21GB RAM. TruncatedSVD dela direktno na sparse in je enakovreden rezultat.
Rezultat se shrani, da ga pri ponovnem zagonu ni treba računati znova.

In [ ]:
npca = 100
if not os.path.exists(pca_file_path):
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(npca, random_state=0)
    combined_data_pca = svd.fit_transform(data_rna)
    pickle.dump(combined_data_pca, open(pca_file_path, 'wb+'))
    pickle.dump(svd, open(pca_file_path.replace('data_rna_pca.pkl', 'rna_pca.pkl'), 'wb+'))
    print(f'TruncatedSVD izračunan in shranjen: {pca_file_path}')
else:
    combined_data_pca = pickle.load(open(pca_file_path, 'rb'))
    print(f'TruncatedSVD naložen iz: {pca_file_path}')

assert combined_data_pca.shape[1] == npca
assert combined_data_pca.shape[0] == data_rna.shape[0]
print(f'RNA PCA oblika: {combined_data_pca.shape}')

## 6. Pripravi trening podatke

Model se trenira na **per-TCR povprečjih** RNA — ne na posameznih celicah.
Vsak TCR klon dobi eno povprečno RNA vrednost (groupby po TCR indeksu).
To zmanjša n_vzorcev iz ~146k celic na ~48k unikatnih TCR.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

x_train = combined_data_pca
x_label = data_labels
assert x_train.shape[0] == x_label.shape[0]

print(f'Pred groupby: {x_train.shape}')
tmp = pd.DataFrame(np.concatenate([x_train, x_label], axis=-1)).groupby(x_train.shape[1] + col_tcr).mean()
x_train = tmp.iloc[:, :x_train.shape[1]].values

tmp2 = tmp.iloc[:, x_train.shape[1]:]
tmp2[tmp.index.name] = tmp.index
tmp2 = tmp2[sorted(tmp2.columns)].values
x_label = tmp2

print(f'Po groupby (per-TCR povprečja): {x_train.shape}  — to je število unikatnih TCR')

## 7. Inicializacija modela

In [ ]:
n_embedding_vocab = 5  # Atchley faktorji: 5 dim/aminokislina
dimz = 100             # velikost TCR embeddinga
batch_size = int(1 * 1024)

G = TCR_Embedder(dim_in=n_embedding_vocab, dimz=dimz, nbase=int(.5 * 1024))
G = G.to(device)

param_list = list(G.parameters())
opt_G = torch.optim.Adam(param_list, lr=.001)

load_train = Loader(x_train, labels=x_label, shuffle=True)
load_eval  = Loader(x_train, labels=x_label, shuffle=False)

print(f'Model parametrov: {sum(p.numel() for p in G.parameters()):,}')
print(f'Batch size: {batch_size}, Training steps: {TRAINING_STEPS}')

## 8. Trening

Loss funkcija ima dve komponenti:
- **MSE rekonstrukcija** (λ=1): kako dobro dekoder rekonstruira originalne Atchley faktorje
- **L2 regularizacija** (λ=0.001): prepreči da embeddingi zrastejo v neskončnost

Pričakovano ~10 min na A100.

In [ ]:
i_iter = 0
losses = []
t = time.time()

while i_iter < TRAINING_STEPS:
    i_iter += 1
    [v.train() for v in [G]]
    opt_G.zero_grad()
    batch_loss = []

    batch_x_rna, batch_labels = load_train.next_batch(batch_size)
    batch_x_tcr = get_x_tcr_from_label(batch_labels)

    batch_x_rna   = numpy2torch(batch_x_rna)
    batch_x_tcr   = numpy2torch(batch_x_tcr)
    batch_labels  = numpy2torch(batch_labels, type=torch.IntTensor)

    batch_embeddings, recon = G(x=batch_x_tcr)

    loss_recon        = ((batch_x_tcr - recon)**2).mean()
    loss_embedding_l2 = (batch_embeddings**2).mean()
    batch_loss = [1 * loss_recon, .001 * loss_embedding_l2]

    batch_loss_list = batch_loss
    total_loss = torch.mean(torch.stack(batch_loss))
    losses.append(total_loss.item())

    total_loss.backward()
    check_for_nan(G)
    opt_G.step()
    opt_G.zero_grad()

    if i_iter % 100 == 0:
        print('{:>5}: avg loss: {:.6f} ({:.1f} s)'.format(i_iter, np.mean(losses), time.time() - t))
        if i_iter % 1000 == 0:
            print('       recon={:.6f}  l2={:.6f}'.format(
                batch_loss_list[0].detach().cpu().numpy(),
                batch_loss_list[1].detach().cpu().numpy()))
        t = time.time()
        losses = []

print('Trening končan!')

## 9. Shrani embeddingi

In [ ]:
[v.eval() for v in [G]]
tcr_embeddings = []
for batch_x_rna, batch_labels in load_eval.iter_batches(batch_size):
    batch_x_tcr = get_x_tcr_from_label(batch_labels)
    batch_x_tcr = numpy2torch(batch_x_tcr)
    batch_embeddings, _ = G(x=batch_x_tcr)
    tcr_embeddings.append(batch_embeddings.detach().cpu().numpy())

tcr_embeddings = np.concatenate(tcr_embeddings, axis=0)

# Skaliranje: če so embeddingi premajhni, jih pomnožimo z 10
if (np.abs(tcr_embeddings).max(axis=0) > 1).sum() < (tcr_embeddings.shape[1] // 2):
    coef = 10
    print('Embeddingi so majhni — skaliranje x10')
else:
    coef = 1
    print('Embeddingi so OK — brez skaliranja')

# trained_tcr_embeddings_ae.pkl: en embedding za vsako unikatno TCR sekvenco
embeddings_to_save = np.zeros([df_all_tcrs.shape[0], tcr_embeddings.shape[1]])
for i in range(x_label.shape[0]):
    row = int(x_label[i, col_tcr])
    embeddings_to_save[row, :] = coef * tcr_embeddings[i]

fn = f'{output_path}/trained_tcr_embeddings_ae.pkl'
with open(fn, 'wb') as f:
    pickle.dump(embeddings_to_save, f)
print(f'trained_tcr_embeddings_ae.pkl shranjen: {embeddings_to_save.shape}')

# data_tcr.pkl: embedding za vsako celico (po TCR indeksu)
combined_data_tcr = np.take(embeddings_to_save, data_labels.values[:, col_tcr].astype(np.int32), axis=0)
with open(f'{output_path}/data_tcr.pkl', 'wb') as f:
    pickle.dump(combined_data_tcr, f)
print(f'data_tcr.pkl shranjen: {combined_data_tcr.shape}  (pričakovano: 146776 × 100)')

## 10. UMAP vizualizacija

In [ ]:
print('Računam UMAP...')
umapper_tcr = umap.UMAP(min_dist=.99, n_neighbors=500)
e_tcr_embeddings = umapper_tcr.fit_transform(tcr_embeddings)

mask_to_plot = np.random.choice(range(combined_data_tcr.shape[0]), min(50000, combined_data_tcr.shape[0]), replace=False)
tcr_embeddings_2 = combined_data_tcr[mask_to_plot]
x_label_2 = data_labels.values[mask_to_plot]
e_tcr_embeddings_2 = umapper_tcr.fit_transform(tcr_embeddings_2)

# Plot po dolžini sekvence
length = np.take((df_all_tcrs_array != 0).sum(axis=-1), x_label[:, col_tcr].astype(np.int32))
r = np.random.permutation(e_tcr_embeddings.shape[0])
fig.clf()
ax = fig.subplots(1, 1)
ax.scatter(e_tcr_embeddings[r, 0], e_tcr_embeddings[r, 1], s=1, c=length[r])
ax.set_title('TCR embedding — barva = dolžina CDR3β')
ax.set_xticks([]); ax.set_yticks([])
fig.savefig(f'{output_path}/tcr_embedding_color_by_length.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot po različnih labelih
cols = [
    (col_celltype, 'SubCellType (0=CD4, 1=CD8)'),
    (col_bloodtumor, 'Tissue (0=Blood, 1=Tumor)'),
    (col_prepost, 'Treatment Stage (0=Pre, 1=Post)'),
    (col_patient, 'Patient'),
]
for col, name in cols:
    labels = x_label_2[:, col]
    unique_labels, color_ids = np.unique(labels, return_inverse=True)
    cmap = plt.get_cmap('tab20', len(unique_labels))
    r = np.random.permutation(e_tcr_embeddings_2.shape[0])
    fig.clf()
    ax = fig.subplots(1, 1)
    ax.scatter(e_tcr_embeddings_2[r, 0], e_tcr_embeddings_2[r, 1], s=2, c=color_ids[r], cmap=cmap)
    for i, lab in enumerate(unique_labels):
        ax.scatter([], [], color=cmap(i), label=str(int(lab)), s=10)
    ax.legend(title=name, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(f'{output_path}/tcr_embedding_color_by_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Vizualizacije shranjene v', output_path)